# Voting Ensemble (Hadi)

Hadi's model from his notebook in PR #24 (`Fraud_Detection_Starter_Hadi.ipynb`), shortened to the structure of `create_xgbc_model.ipynb`. Same team split as the other models; all lookup tables (seasonal benchmark, region and tariff risk) are learned on the train part only.

Fixes compared to the original: every member is tuned inside the ensemble (the unlimited decision tree and KNN with k=5 were close to a coin flip on their own), and HistGradientBoosting – Hadi's best single model – joins the vote.

In [ ]:
# Setup root path one folder up
from pathlib import Path
import sys
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
MODEL_PATH = Path.cwd() / "voting_model.joblib"

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import helper as h
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Prepare Data

In [ ]:
# Load the raw tables
clients = pd.read_csv(ROOT / "data/train/client_train.csv")
invoices = pd.read_csv(ROOT / "data/train/invoice_train.csv", low_memory=False)

In [ ]:
# Stratified train/holdout split at client level (same team split as the other models)
train_clients, holdout_clients = train_test_split(
    clients,
    test_size=0.2,
    stratify=clients["target"],
    random_state=42,
)

train_invoices = invoices[invoices["client_id"].isin(train_clients["client_id"])]
holdout_invoices = invoices[invoices["client_id"].isin(holdout_clients["client_id"])]

# Build features

In [ ]:
# Hadi's features: bill-level consumption per month, then one row per client
LEVELS = ["consommation_level_1", "consommation_level_2", "consommation_level_3"]


def prepare_bills(clients, invoices):
    """Bills of at most 12 months with monthly consumption, region and client category attached."""
    inv = invoices.merge(clients[["client_id", "region", "client_catg"]], on="client_id", how="inner")
    inv = inv[(inv["months_number"] > 0) & (inv["months_number"] <= 12)].copy()  # > 0 avoids division by zero
    inv["invoice_month"] = pd.to_datetime(inv["invoice_date"]).dt.month
    inv["monthly_consumption"] = (inv["new_index"] - inv["old_index"]) / inv["months_number"]
    return inv


def fit_lookups(clients, bills):
    """Seasonal benchmark and region / tariff risk groups – learned on the train part only."""
    month_mean = bills.groupby(["region", "client_catg", "counter_type", "invoice_month"])["monthly_consumption"].mean()
    s = month_mean.groupby(["region", "client_catg", "counter_type"]).agg(["max", "min", "mean"])
    seasonal = ((s["max"] - s["min"]) / s["mean"]).rename("seasonal_benchmark").reset_index()

    target = clients.set_index("client_id")["target"]
    tarif = h.most_common(bills, "tarif_type")
    region_rate = target.groupby(clients.set_index("client_id")["region"]).mean()
    tarif_rate = target.reindex(tarif.index).groupby(tarif).mean()
    region_risk = pd.qcut(region_rate, q=4, labels=False, duplicates="drop")
    tarif_risk = pd.qcut(tarif_rate, q=4, labels=False, duplicates="drop")
    return seasonal, region_risk, tarif_risk


def build_hadi_features(clients, bills, lookups):
    """One row per client (all clients, NaN where a client has no usable bill)."""
    seasonal, region_risk, tarif_risk = lookups
    key = ["client_id", "counter_type"]

    # consumption range per client and meter type: (avg of 3 largest - avg of 3 smallest) / mean
    b = bills[key + ["monthly_consumption"]].dropna().sort_values("monthly_consumption")
    low = b.groupby(key).head(3).groupby(key)["monthly_consumption"].mean()
    high = b.groupby(key).tail(3).groupby(key)["monthly_consumption"].mean()
    mean = b.groupby(key)["monthly_consumption"].mean()
    rng = ((high - low) / mean.replace(0, np.nan)).rename("consumption_range").reset_index()

    # variability index = own range / seasonal range of the peer group
    v = bills[key + ["region", "client_catg"]].drop_duplicates(key).merge(rng, on=key, how="left")
    v = v.merge(seasonal, on=["region", "client_catg", "counter_type"], how="left")
    v["variability_index"] = v["consumption_range"] / v["seasonal_benchmark"].replace(0, np.nan)

    cl = clients.set_index("client_id")
    feats = pd.DataFrame(index=cl.index)
    for col in ["counter_statue", "counter_code", "counter_coefficient"]:
        feats[col] = pd.to_numeric(h.most_common(bills, col), errors="coerce")
    feats = feats.join(bills.groupby("client_id")[LEVELS + ["monthly_consumption"]].mean())
    feats["variability_index"] = v.groupby("client_id")["variability_index"].mean()
    feats["client_catg"] = cl["client_catg"]

    # risk groups as dummies, group 0 = reference (Hadi: "two variables to dummy")
    reg = cl["region"].map(region_risk)
    tar = h.most_common(bills, "tarif_type").map(tarif_risk).reindex(cl.index)
    for k in (1, 2, 3):
        feats[f"region_risk_{k}"] = (reg == k).astype(int)
        feats[f"tarif_risk_{k}"] = (tar == k).astype(int)
    return feats

In [ ]:
# Build the feature tables for train and holdout (lookups learned on train only)
train_bills = prepare_bills(train_clients, train_invoices)
holdout_bills = prepare_bills(holdout_clients, holdout_invoices)
lookups = fit_lookups(train_clients, train_bills)

X_train = build_hadi_features(train_clients, train_bills, lookups)
X_holdout = build_hadi_features(holdout_clients, holdout_bills, lookups)
y_train = train_clients.set_index("client_id")["target"].reindex(X_train.index)
y_holdout = holdout_clients.set_index("client_id")["target"].reindex(X_holdout.index)

# Self-check: one row per client, same columns in both sets
for name, X, cl in [("train", X_train, train_clients), ("holdout", X_holdout, holdout_clients)]:
    assert X.index.is_unique and len(X) == len(cl), name
    print(f"{name:8s} {X.shape}")
assert list(X_train.columns) == list(X_holdout.columns)

# Training and Export

In [ ]:
# Soft-voting ensemble of 5 tuned models using VotingClassifier. 
# Ensemble of Logistic-Regression, K-Neighbors-Classifier, Decision-Tree, RandomForest and Gradient-Boosting
def pipe(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps + [("model", model)])

voting = VotingClassifier(
    estimators=[
        ("lr", pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), scale=True)),
        ("knn", pipe(KNeighborsClassifier(n_neighbors=51, weights="distance"), scale=True)),   # k=5 alone: 0.58
        ("dt", pipe(DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=42,
                                           class_weight="balanced"))),                       # unlimited depth alone: 0.52
        ("rf", pipe(RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=42,
                                           class_weight="balanced", n_jobs=1))),
        ("hgb", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=31,
                                               l2_regularization=1.0, random_state=42)),
    ],
    voting="soft",
)
# Examination of different Weights 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    voting,
    param_grid={"weights": [[1, 1, 1, 1, 1], [1, 1, 1, 2, 2], [1, 1, 1, 2, 3]]},
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train, y_train)

voting_model = grid.best_estimator_
best_idx = grid.best_index_
cv_auc_mean = grid.cv_results_["mean_test_score"][best_idx]
cv_auc_std = grid.cv_results_["std_test_score"][best_idx]
print(f"Best CV ROC-AUC: {cv_auc_mean:.4f} (+/- {cv_auc_std:.4f})  weights {grid.best_params_['weights']}")

In [ ]:
# Export trained model
joblib.dump(voting_model, MODEL_PATH, compress=3)

# Metrics and Plots

In [ ]:
# Use import if not interested in training
voting_model = joblib.load(MODEL_PATH)

In [ ]:
# Holdout ROC-AUC and business value on the top 10 % most probable frauds
train_proba = voting_model.predict_proba(X_train)[:, 1]
test_proba = voting_model.predict_proba(X_holdout)[:, 1]
holdout_auc = roc_auc_score(y_holdout, test_proba)
print(f"Holdout ROC-AUC: {holdout_auc:.4f}")

business_report = h.top_fraction_report(y_holdout, test_proba, frac=0.10)
print("Business value (top 10% most probable frauds, holdout):")
for metric, value in business_report.items():
    print(f"  {metric:10s} {value:.4f}" if isinstance(value, float) else f"  {metric:10s} {value}")

# Save the result for the main notebook (5.5) -> models/results.csv
h.save_result("Voting Ensemble", "Hadi", "models/create_voting_model.ipynb", cv_auc_mean, cv_auc_std,
              business_report["recall"], business_report["precision"], holdout_auc)

In [ ]:
# Train vs. holdout ROC-AUC (a big gap = the model learned the training clients by heart)
scores = pd.Series({"Train": roc_auc_score(y_train, train_proba), "Test": holdout_auc})
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(scores.index, scores.values, color=["#4C72B0", "#DD8452"])
ax.bar_label(bars, fmt="%.3f", padding=2)
ax.set_ylim(0, 1.1)
ax.set_ylabel("ROC-AUC")
ax.set_title("Voting Ensemble: train vs. test")
plt.tight_layout()
plt.show()